<a href="https://colab.research.google.com/github/lamia-khd/react-router/blob/master/calcule2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time # Imports the built-in module used to measure elapsed time
import numpy as np # Imports the NumPy library for fast numerical operations
from concurrent.futures import ThreadPoolExecutor # Imports the tool to manage multi-threading

def read_config(): # Defines the function that simulates reading the initial settings
    """Settings file: read first, by one worker only (2 s).""" # Documentation string for the setup function
    time.sleep(2) # Pauses the program for 2 seconds to simulate sequential read time

def download(seconds): # Defines the function simulating the file download process
    """Download one shop's sales file (it takes some seconds).""" # Documentation string for the task
    time.sleep(seconds) # Pauses for the amount of seconds passed as an argument

files = [1, 1, 1, 1, 1, 1, 1, 1] # List representing today's 8 files, each taking 1 second to download
tomorrow = [4, 1, 1, 1, 1, 1, 1, 2] # List representing tomorrow's files with varying download times
rng = np.random.default_rng(0) # Initializes the random number generator with a fixed seed
amounts = rng.integers(100, 5000, 2_000_000) # Generates an array of 2 million random sales figures

# Q1
t0 = time.perf_counter() # Starts the stopwatch for the single-worker execution
read_config() # Executes the mandatory 2-second sequential configuration read
for f in files: # Loops through every file in today's list one by one
    download(f) # Downloads the current file in the sequence
T1 = time.perf_counter() - t0 # Calculates the total execution time for 1 worker
print(f"Q1 1 worker: {T1:.1f} s") # Prints the resulting time

# Q2
t0 = time.perf_counter() # Resets and starts the stopwatch for the parallel execution
read_config() # Executes the sequential configuration step again
with ThreadPoolExecutor(max_workers=4) as team: # Creates a team of 4 independent workers
    list(team.map(download, files)) # Assigns the download tasks for the 8 files to the team
Tp = time.perf_counter() - t0 # Calculates the total execution time with 4 workers
print(f"Q2 4 workers: {Tp:.1f} s, speed-up {T1 / Tp:.1f}") # Prints the time and the calculated speed-up

# Q3
part_alone = 2.0 # Defines the time taken by the indivisible sequential portion
print(f"Q3 alone: {part_alone:.1f} s, best possible: {T1 / part_alone:.1f}") # Prints max theoretical speed-up

# Q4
t_loop = time.perf_counter() # Starts the timer for the standard Python loop addition
total_loop = sum(amounts) # Calculates the sum using Python's built-in sum function
loop_time = time.perf_counter() - t_loop # Records the time taken by the standard loop

t_numpy = time.perf_counter() # Starts the timer for the optimized NumPy addition
total_numpy = amounts.sum() # Calculates the sum using NumPy's highly optimized SIMD method
numpy_time = time.perf_counter() - t_numpy # Records the time taken by NumPy
print(f"Q4 loop {loop_time:.4f} s, numpy {numpy_time:.4f} s, same total: {total_loop == total_numpy}") # Prints comparison

# Q5
t0 = time.perf_counter() # Starts the stopwatch for tomorrow's uneven schedule
read_config() # Executes the required 2-second setup
with ThreadPoolExecutor(max_workers=4) as team: # Creates a team of 4 workers for the new workload
    list(team.map(download, tomorrow)) # Executes tomorrow's varied tasks in parallel
print(f"Q5 tomorrow, 4 workers: {time.perf_counter() - t0:.1f} s") # Prints the actual completion time

Q1 1 worker: 10.0 s
Q2 4 workers: 4.0 s, speed-up 2.5
Q3 alone: 2.0 s, best possible: 5.0
Q4 loop 0.1574 s, numpy 0.0016 s, same total: True
Q5 tomorrow, 4 workers: 6.0 s


In [ ]:

T1 = 100      # Total training time on one core, in minutes
alone = 5     # Time that cannot be parallelized
T8 = 16.9     # Measured time on the 8-core server
T64 = 6.5     # Measured time on the 64-core server

# Q1: Calculate the speed-up on the 8-core server
speedup8 = T1 / T8
print(f"Q1 8 cores: x{speedup8:.2f}")

# Q2: Calculate the speed-up on the 64-core server
speedup64 = T1 / T64
print(f"Q2 64 cores: x{speedup64:.2f}")

# Q3: Compare the speed-up of the two servers
comparison = T8 / T64
print(f"Q3 big / small: x{comparison:.2f}")

# Q4: Calculate the best possible speed-up and training time
# The 5-minute part cannot be parallelized
best_time = alone
best_speedup = T1 / best_time
print(f"Q4 best possible: x{best_speedup:.2f}, {best_time:.2f} min")

# Q5: Calculate the new best speed-up if the non-parallelizable
# part is reduced to 1 minute
new_alone = 1
new_best_speedup = T1 / new_alone
print(f"Q5 alone = 1 min: best x{new_best_speedup:.2f}")

Q1 8 cores: x5.92
Q2 64 cores: x15.38
Q3 big / small: x2.60
Q4 best possible: x20.00, 5.00 min
Q5 alone = 1 min: best x100.00


### Detailed Summary of Results

#### Section 1: Parallelization and SIMD
* **Q1 (1 Worker - Sequential):** `10.0 s` — Includes a mandatory $2	ext{ s}$ sequential configuration initialization and $8	ext{ s}$ total downloading 8 files sequentially ($1	ext{ s}$ each).
* **Q2 (4 Workers - Parallel):** `4.0 s` (Speed-up: `2.5x`) — The $2	ext{ s}$ setup remains sequential, while the 8 download tasks are processed across 4 workers in parallel (taking $2	ext{ s}$ total).
* **Q3 (Amdahl's Limit):** `5.0x` maximum speed-up — Since the $2	ext{ s}$ setup cannot be parallelized, the execution time can never drop below $2	ext{ s}$, limiting the potential speed-up to $10	ext{ s} / 2	ext{ s} = 5.0	ext{x}$.
* **Q4 (Loop vs SIMD Addition):** NumPy processes the $2	ext{ million}$ elements in `~0.0016 s` compared to standard Python's `~0.1574 s` (~100x speedup) by bypassing interpreter overhead and utilizing vectorized CPU hardware instructions (SIMD).
* **Q5 (Tomorrow's Schedule):** `6.0 s` — With tasks of uneven lengths (`[4, 1, 1, 1, 1, 1, 1, 2]`), the parallel downloading duration is bottlenecked by the longest single task ($4	ext{ s}$), resulting in $2	ext{ s} + 4	ext{ s} = 6.0	ext{ s}$ total.

#### Section 2: Amdahl's Law and Scaling
* **Q1 (8 Cores):** `x5.92` speed-up — Derived from dividing the single-core training time by the 8-core time ($100	ext{ min} / 16.9	ext{ min}$).
* **Q2 (64 Cores):** `x15.38` speed-up — Calculated as $100	ext{ min} / 6.5	ext{ min}$, illustrating diminishing returns as more cores are added to a system with a fixed serial bottleneck.
* **Q3 (Relative Performance):** `x2.60` relative improvement — Compares the performance of the 64-core server directly to the 8-core server ($16.9	ext{ min} / 6.5	ext{ min}$).
* **Q4 (Theoretical Limits):** Max Speed-up `x20.00` / Minimum Time `5.00 min` — Because $5	ext{ min}$ of training cannot be parallelized, the run time cannot drop below $5	ext{ min}$ ($100	ext{ min} / 5	ext{ min}$).
* **Q5 (Optimized Serial Portion):** Max Speed-up `x100.00` — Reducing the non-parallelizable step from $5	ext{ min}$ to $1	ext{ min}$ dramatically raises the theoretical scaling roof ($100	ext{ min} / 1	ext{ min}$).

# tp3

In [ ]:
import threading, time # Importation des modules pour le multithreading et le temps

# --- Q1 (Sans verrou) ---
likes = 0 # Initialisation du compteur partagé à zéro

def add_likes(n): # Définition de la fonction ajoutant n likes
    global likes # Indication que nous modifions la variable globale[cite: 2]
    for _ in range(n): # Boucle s'exécutant n fois[cite: 2]
        current = likes # Étape 1 : Lecture non protégée de la valeur actuelle[cite: 2]
        time.sleep(0.0001) # Simulation de la charge du serveur via une courte pause[cite: 2]
        likes = current + 1 # Étape 2 : Écriture de la nouvelle valeur incrémentée[cite: 2]

threads_q1 = [] # Création d'une liste pour suivre nos threads
for _ in range(4): # Préparation de 4 threads comme demandé[cite: 2]
    t = threading.Thread(target=add_likes, args=(50,)) # Assignation de 50 likes par serveur[cite: 2]
    threads_q1.append(t) # Ajout du thread à la liste de suivi
    t.start() # Démarrage du thread pour exécution simultanée[cite: 2]

for t in threads_q1: # Parcours de chaque thread lancé
    t.join() # Attente de la complétion de chaque exécution[cite: 2]

print(f"Q1 likes: {likes}  (expected 200)") # Affichage du résultat, qui sera erroné[cite: 2]

# --- Q2 (Avec verrou) ---
likes = 0 # Réinitialisation du compteur pour l'essai sécurisé
lock = threading.Lock() # Création d'un verrou d'exclusion mutuelle[cite: 2]

def add_likes_safe(n): # Création d'une nouvelle fonction sécurisée
    global likes # Référence au même compteur global
    for _ in range(n): # Boucle de traitement répétée
        with lock: # Pose du verrou englobant lecture et écriture[cite: 2]
            current = likes # Lecture sécurisée bloquant les autres threads
            time.sleep(0.0001) # Pause sans risque d'interruption concurrente
            likes = current + 1 # Écriture sécurisée validant l'incrément

threads_q2 = [] # Nouvelle liste pour la deuxième série de threads
for _ in range(4): # Création des 4 nouveaux serveurs virtuels
    t = threading.Thread(target=add_likes_safe, args=(50,)) # Utilisation de la fonction avec verrou
    threads_q2.append(t) # Ajout au gestionnaire
    t.start() # Lancement des opérations sécurisées

for t in threads_q2: # Boucle d'attente globale
    t.join() # Synchronisation finale

print(f"Q2 likes: {likes}") # Affichage du résultat exact attendu[cite: 2]

Q1 likes: 50  (expected 200)
Q2 likes: 200


### Why were likes lost in Q1? (Race Condition)

In **Q1**, the threads run simultaneously without any synchronization (no locks):

1. **Overlapping Reads:** Multiple threads read the same value of `likes` concurrently (e.g., they all read `0` before any of them write the update).
2. **Context Switch:** During `time.sleep()`, threads pause while holding their outdated local copy.
3. **Overwriting Writes:** When resuming, they all perform `0 + 1 = 1` and write `1` back. This overwrites each other's increments, causing the lost updates.

In [ ]:
import threading, time

# ---- Q1 & Q4 (Sans verrou) ----
seats = 100
sold = 0

def sell(n):
    global seats, sold
    for _ in range(n):
        if seats > 0:
            time.sleep(0.001)
            seats -= 1
            sold += 1

start_time_no_lock = time.time()
threads = []
for _ in range(4):
    t = threading.Thread(target=sell, args=(30,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()
time_no_lock = time.time() - start_time_no_lock

print(f"Q1 sold: {sold}, seats left: {seats}")

# ---- Q3 & Q4 (Avec verrou) ----
seats = 100  # Réinitialisation des variables partagées
sold = 0
lock = threading.Lock()

def sell_safe(n):
    global seats, sold
    for _ in range(n):
        with lock:  # Le verrou protège la section critique
            if seats > 0:
                time.sleep(0.001)
                seats -= 1
                sold += 1

start_time_lock = time.time()
threads_safe = []
for _ in range(4):
    t = threading.Thread(target=sell_safe, args=(30,))
    threads_safe.append(t)
    t.start()

for t in threads_safe:
    t.join()
time_lock = time.time() - start_time_lock

print(f"Q3 sold: {sold}, seats left: {seats}")
print(f"Q4 time: {time_no_lock:.4f} s without lock, {time_lock:.4f} s with lock")

Q1 sold: 103, seats left: -3
Q3 sold: 100, seats left: 0
Q4 time: 0.0321 s without lock, 0.1191 s with lock


In [ ]:
import time  # Importation du module time pour chronométrer l'exécution
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor  # Importation des gestionnaires de threads et processus

def count_primes(block):  # Définition de la fonction qui compte les nombres premiers dans un intervalle
    lo, hi = block  # Extraction des bornes inférieure et supérieure depuis le tuple 'block'
    n = 0  # Initialisation du compteur de nombres premiers à zéro
    for x in range(max(lo, 2), hi):  # Boucle sur chaque entier de l'intervalle (en commençant au moins à 2)
        d, prime = 2, True  # Initialisation du diviseur à 2 et du drapeau de primalité à Vrai
        while d * d <= x:  # Boucle de vérification jusqu'à la racine carrée du nombre 'x'
            if x % d == 0:  # Vérification si 'x' est divisible par 'd' sans reste
                prime = False  # Si divisible, le nombre n'est pas premier, on change le drapeau
                break  # On interrompt la boucle 'while' car on a trouvé un diviseur
            d += 1  # Incrémentation du diviseur pour tester le prochain nombre
        if prime:  # Si le drapeau est toujours Vrai après avoir testé tous les diviseurs possibles
            n += 1  # On incrémente le compteur total de nombres premiers trouvés
    return n  # Retour du nombre total de nombres premiers pour ce bloc

if __name__ == '__main__':  # Protection indispensable sous Windows pour utiliser ProcessPoolExecutor
    blocks = [(0, 100_000), (100_000, 200_000), (200_000, 300_000), (300_000, 400_000)]  # Définition des 4 blocs de calcul

    # --- Q1 : Calcul séquentiel (un par un) ---
    start_q1 = time.time()  # Mémorisation de l'heure de début pour la question 1
    total_primes = sum(count_primes(b) for b in blocks)  # Calcul de la somme des premiers en itérant sur chaque bloc séquentiellement
    time_q1 = time.time() - start_q1  # Calcul de la durée totale d'exécution de la méthode séquentielle
    print(f"Q1 primes: {total_primes}, one by one: {time_q1:.4f} s")  # Affichage du nombre total et du temps pris

    # --- Q2 : Calcul avec 4 threads ---
    start_q2 = time.time()  # Mémorisation de l'heure de début pour la question 2
    with ThreadPoolExecutor(max_workers=4) as team:  # Création d'un groupe d'exécution de 4 threads
        list(team.map(count_primes, blocks))  # Exécution de la fonction sur les blocs en utilisant les threads
    time_q2 = time.time() - start_q2  # Calcul de la durée d'exécution avec les threads
    print(f"Q2 4 threads: {time_q2:.4f} s")  # Affichage du temps pris par l'exécution multi-thread

    # --- Q3 : Calcul avec 4 processus ---
    start_q3 = time.time()  # Mémorisation de l'heure de début pour la question 3
    with ProcessPoolExecutor(max_workers=4) as team:  # Création d'un groupe d'exécution de 4 processus distincts
        list(team.map(count_primes, blocks))  # Exécution parallèle de la fonction sur les blocs via des processus
    time_q3 = time.time() - start_q3  # Calcul de la durée d'exécution avec les processus
    speed_up = time_q1 / time_q3  # Calcul du facteur d'accélération (temps séquentiel divisé par temps multiprocessing)
    print(f"Q3 4 processes: {time_q3:.4f} s, speed-up {speed_up:.2f}x")  # Affichage du temps pris et de l'accélération obtenue

Q1 primes: 33860, one by one: 4.2202 s
Q2 4 threads: 2.8938 s
Q3 4 processes: 2.5376 s, speed-up 1.66x


homework

In [1]:
# Importer la bibliothèque time pour chronométrer l'exécution
import time
# Importer les outils de concurrence pour les threads et les processus
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

# Définition de la fonction de téléchargement donnée
def download(shop):
    # Simuler le temps de réponse du serveur d'une seconde
    time.sleep(1)
    # Retourner l'identifiant de la boutique
    return shop

# Définition de la fonction d'analyse donnée
def analyse(shop):
    # Initialiser la variable totale à zéro
    total = 0
    # Boucle pour simuler un calcul lourd avec dix millions d'itérations
    for k in range(10_000_000):
        # Effectuer des opérations mathématiques basiques
        total += (k * shop) % 7
    # Retourner le résultat total du calcul
    return total

# Point d'entrée principal requis sous Windows pour le multiprocessing
if __name__ == '__main__':
    # Liste des quatre boutiques à traiter
    shops = [1, 2, 3, 4]

    # --- Q1 : Exécution séquentielle ---
    # Démarrer le chronomètre pour la question 1
    start_q1 = time.time()
    # Créer une liste vide pour stocker les résultats
    results_q1 = []
    # Parcourir chaque boutique une par une
    for shop in shops:
        # Télécharger la donnée de la boutique courante
        data = download(shop)
        # Analyser la donnée téléchargée
        result = analyse(data)
        # Ajouter le résultat final à notre liste
        results_q1.append(result)
    # Calculer la durée totale de l'exécution séquentielle
    time_q1 = time.time() - start_q1
    # Afficher le résultat formaté selon les consignes
    print(f"Q1 one by one: {time_q1:.2f} s")

    # --- Q2 : Téléchargement avec le bon outil ---
    # Démarrer le chronomètre pour la question 2
    start_q2 = time.time()
    # Initialiser un pool de threads car c'est une tâche d'attente (I/O)
    with ThreadPoolExecutor(max_workers=4) as executor:
        # Lancer les téléchargements en parallèle pour les 4 boutiques
        downloads = list(executor.map(download, shops))
    # Calculer la durée du téléchargement multithread
    time_q2 = time.time() - start_q2
    # Afficher le temps pris par les 4 threads
    print(f"Q2 download with 4 threads: {time_q2:.2f} s")

    # --- Q3 : Analyse avec threads puis processus ---
    # Démarrer le chrono pour l'analyse avec les threads
    start_q3_t = time.time()
    # Créer un pool de threads pour essayer de paralléliser le calcul
    with ThreadPoolExecutor(max_workers=4) as executor:
        # Lancer l'analyse lourde dans les threads
        res_t = list(executor.map(analyse, downloads))
    # Enregistrer la durée de l'analyse via threads
    time_q3_t = time.time() - start_q3_t

    # Démarrer le chrono pour l'analyse avec les processus
    start_q3_p = time.time()
    # Créer un pool de processus pour utiliser plusieurs cœurs CPU
    with ProcessPoolExecutor(max_workers=4) as executor:
        # Lancer l'analyse lourde dans des processus séparés
        res_p = list(executor.map(analyse, downloads))
    # Enregistrer la durée de l'analyse via processus
    time_q3_p = time.time() - start_q3_p
    # Afficher et comparer les deux temps obtenus
    print(f"Q3 analyse: threads {time_q3_t:.2f} s, processes {time_q3_p:.2f} s")

    # --- Q4 : Le meilleur rapport (Meilleur téléchargement + Meilleure analyse) ---
    # Démarrer le chronomètre pour la solution optimale
    start_q4 = time.time()
    # Utiliser les threads pour le téléchargement (attente réseau)
    with ThreadPoolExecutor(max_workers=4) as executor:
        # Exécuter les téléchargements en parallèle
        best_dl = list(executor.map(download, shops))
    # Utiliser les processus pour l'analyse (calcul intensif)
    with ProcessPoolExecutor(max_workers=4) as executor:
        # Exécuter les calculs sur différents cœurs du processeur
        best_res = list(executor.map(analyse, best_dl))
    # Calculer la durée de la meilleure approche combinée
    time_q4 = time.time() - start_q4
    # Calculer l'accélération par rapport à la méthode séquentielle de Q1
    speedup = time_q1 / time_q4
    # Afficher le temps optimal et le multiplicateur de vitesse
    print(f"Q4 best report: {time_q4:.2f} s, speed-up x{speedup:.2f}")

    # --- Q5 : Vérification des résultats ---
    # Comparer les résultats de Q1 avec ceux de Q4
    is_same = (results_q1 == best_res)
    # Afficher si les résultats sont identiques (True/False)
    print(f"Q5 same results: {is_same}")

Q1 one by one: 10.56 s
Q2 download with 4 threads: 1.00 s
Q3 analyse: threads 4.90 s, processes 4.67 s
Q4 best report: 7.42 s, speed-up x1.42
Q5 same results: True


### Quick Summary (TP3 Homework)

* **Q3 Analysis Times:** Threads (`~4.90 s`) vs Processes (`~4.67 s`). Processes are faster because analyzing is **CPU-bound** (10M iterations). Python's **GIL** limits multi-threading to a single core, whereas processes run on multiple CPU cores in parallel.
* **Q4 Best Report:** Time is `~7.42 s` with a **speed-up of x1.42** over sequential (`10.56 s`). This combines the best of both: threads for downloading and processes for analysis.
* **Q5 Results & Tool Choices:**
  * **Same results?** Yes (`True`).
  * **Download (Threads):** **I/O-bound** (waiting on network). Threads are lightweight and release control during idle wait times.
  * **Analyse (Processes):** **CPU-bound** (heavy math). Processes bypass the GIL to fully use multi-core CPUs.